# Airbnb Pricing Analysis — European Cities
## Data Cleaning & Feature Engineering with Python

This notebook combines, cleans and prepares 20 raw CSV files from the
[Airbnb Prices in European Cities](https://www.kaggle.com/datasets/thedevastator/airbnb-prices-in-european-cities) dataset.

**Cities covered:** Amsterdam, Athens, Barcelona, Berlin, Budapest, Lisbon, London, Paris, Rome, Vienna

**Output:** A single clean CSV file ready for Power BI and machine learning.

> **Note:** This notebook runs on Google Colab. Upload all 20 CSV files to your Google Drive and update the folder path in the first code cell.


## 1. Mount Google Drive

We mount Google Drive to access the dataset files stored in the cloud.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load & Combine Raw Files

### 2.1 Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)


### 2.2 Combine All CSV Files

Each file follows the naming pattern `city_weekdays.csv` or `city_weekends.csv`.
We extract the city name and day type from the filename and add them as columns.


In [ ]:
# Define the folder where CSV files are located
FOLDER = '/content/drive/MyDrive/Portfolio/Colab_notebooks/Project3/Dataset/'

files = [f for f in glob.glob(FOLDER + '*.csv') if 'weekdays' in f or 'weekends' in f]
dfs = []

day_type_map = {'weekdays': 'Weekday', 'weekends': 'Weekend'}

for f in files:
    name = os.path.basename(f).replace('.csv', '')
    # Remove numeric prefix if present (e.g. '1776248233563_amsterdam_weekdays')
    parts = name.split('_')
    if parts[0].isdigit():
        parts = parts[1:]
    day_type = parts[-1]         # 'weekdays' or 'weekends'
    city = '_'.join(parts[:-1])  # city name
    df = pd.read_csv(f)
    df['city'] = city.capitalize()
    df['day_type'] = day_type_map.get(day_type, day_type)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
print(f'Combined dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Cities: {sorted(df["city"].unique())}')
print(f'Day types: {df["day_type"].unique().tolist()}')

Combined dataset: 51,707 rows, 22 columns
Cities: ['Amsterdam', 'Athens', 'Barcelona', 'Berlin', 'Budapest', 'Lisbon', 'London', 'Paris', 'Rome', 'Vienna']
Day types: ['Weekend', 'Weekday']


## 3. Data Overview

### 3.1 Structure and Data Types

We examine the number of rows, columns and the data type of each field.


In [ ]:
print(f'Shape: {df.shape}')
print()
print(df.dtypes)


Shape: (51707, 22)

Unnamed: 0                      int64
realSum                       float64
room_type                      object
room_shared                      bool
room_private                     bool
person_capacity               float64
host_is_superhost                bool
multi                           int64
biz                             int64
cleanliness_rating            float64
guest_satisfaction_overall    float64
bedrooms                        int64
dist                          float64
metro_dist                    float64
attr_index                    float64
attr_index_norm               float64
rest_index                    float64
rest_index_norm               float64
lng                           float64
lat                           float64
city                           object
day_type                       object
dtype: object


### 3.2 First Rows

We review the first rows to understand the content and structure of the dataset.

In [ ]:
df.head()


,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,0,319.640053,Private room,False,True,2.0,False,0,1,9.0,...,4.763360,0.852117,110.906123,5.871971,136.982208,11.941560,4.84639,52.34137,Amsterdam,Weekend
1,1,347.995219,Private room,False,True,2.0,False,0,1,9.0,...,5.748310,3.651591,75.275937,3.985516,95.386468,8.315410,4.97512,52.36103,Amsterdam,Weekend
2,2,482.975183,Private room,False,True,4.0,False,0,1,9.0,...,0.384872,0.439852,493.272517,26.116521,875.114817,76.289005,4.89417,52.37663,Amsterdam,Weekend
3,3,485.552926,Private room,False,True,2.0,True,0,0,10.0,...,0.544723,0.318688,552.849514,29.270850,815.303994,71.074937,4.90051,52.37508,Amsterdam,Weekend
4,4,2771.541724,Entire home/apt,False,False,4.0,True,0,0,10.0,...,1.686798,1.458399,208.809162,11.055489,272.315202,23.739349,4.88467,52.38749,Amsterdam,Weekend


### 3.3 Missing Values

We check how many null values exist in each column.

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')


Missing values per column:
No missing values found.


### 3.4 Duplicate Rows

We check for duplicate rows that could bias the analysis.


In [ ]:
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')


Duplicate rows: 0


## 4. Data Cleaning

### 4.1 Drop Unnecessary Columns

We remove the following columns:
- `Unnamed: 0` — original row index, no analytical value.
- `attr_index` and `rest_index` — raw scores already represented by their normalized versions.


In [ ]:
df = df.drop(columns=['Unnamed: 0', 'attr_index', 'rest_index'])
print(f'Columns after drop: {df.shape[1]}')
print(df.columns.tolist())


Columns after drop: 19
['realSum', 'room_type', 'room_shared', 'room_private', 'person_capacity', 'host_is_superhost', 'multi', 'biz', 'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist', 'attr_index_norm', 'rest_index_norm', 'lng', 'lat', 'city', 'day_type']


### 4.2 Rename Columns

We rename all columns to be more descriptive and consistent.

In [ ]:
df = df.rename(columns={
    'realSum':                    'price',
    'room_type':                  'room_type',
    'room_shared':                'is_shared_room',
    'room_private':               'is_private_room',
    'person_capacity':            'person_capacity',
    'host_is_superhost':          'is_superhost',
    'multi':                      'host_has_multiple_listings',
    'biz':                        'host_is_business',
    'cleanliness_rating':         'cleanliness_rating',
    'guest_satisfaction_overall': 'overall_rating',
    'bedrooms':                   'bedrooms',
    'dist':                       'dist_to_center_km',
    'metro_dist':                 'dist_to_metro_km',
    'attr_index_norm':            'attraction_index',
    'rest_index_norm':            'restaurant_index',
    'lng':                        'longitude',
    'lat':                        'latitude',
})

print('Columns after rename:')
print(df.columns.tolist())

Columns after rename:
['city', 'day_type', 'room_type', 'is_shared_room', 'is_private_room', 'is_superhost', 'host_has_multiple_listings', 'host_is_business', 'person_capacity', 'bedrooms', 'bedroom_label', 'price', 'price_per_night', 'cleanliness_rating', 'overall_rating', 'dist_to_center_km', 'dist_to_metro_km', 'attraction_index', 'restaurant_index', 'longitude', 'latitude']


### 4.3 Fix Data Types

We convert columns to their correct data types:
- `person_capacity` is float — we convert it to int since you can't have 2.5 people.
- `is_superhost`, `is_shared_room`, `is_private_room` are bool — we convert to int (0/1) for compatibility with Power BI and machine learning models.

In [ ]:
df['person_capacity'] = df['person_capacity'].astype(int)
df['is_superhost'] = df['is_superhost'].astype(int)
df['is_shared_room'] = df['is_shared_room'].astype(int)
df['is_private_room'] = df['is_private_room'].astype(int)
df['latitude'] = df['latitude'].round(2)
df['longitude'] = df['longitude'].round(2)

print('Data types after conversion:')
print(df.dtypes)

Data types after conversion:
city                           object
day_type                       object
room_type                      object
is_shared_room                  int64
is_private_room                 int64
is_superhost                    int64
host_has_multiple_listings      int64
host_is_business                int64
person_capacity                 int64
bedrooms                        int64
bedroom_label                  object
price                         float64
price_per_night               float64
cleanliness_rating            float64
overall_rating                float64
dist_to_center_km             float64
dist_to_metro_km              float64
attraction_index              float64
restaurant_index              float64
longitude                     float64
latitude                      float64
dtype: object


### 4.4 Outlier Analysis — Price

The `price` column represents the total price for 2 nights and 2 people.
We calculate the price per night per person and remove extreme outliers above €500.

In [ ]:
# Calculate price per night per person
df['price_per_night'] = (df['price'] / 4).round(2)

print('Price per night per person — stats:')
print(df['price_per_night'].describe().round(2))
print(f'\nListings above 500 EUR/night: {(df["price_per_night"] > 500).sum()}')

Price per night per person — stats:
count    51707.00
mean        69.97
std         81.99
min          8.69
25%         37.19
50%         52.84
75%         79.92
max       4636.36
Name: price_per_night, dtype: float64

Listings above 500 EUR/night: 136


In [ ]:
# Remove outliers above 500 EUR per night per person
before = len(df)
df = df[df['price_per_night'] <= 500].copy()
after = len(df)
print(f'Rows removed: {before - after}')
print(f'Rows remaining: {after:,}')


Rows removed: 136
Rows remaining: 51,571


## 5. Feature Engineering

### 5.1 Bedroom Labels

We create a descriptive label for the number of bedrooms to help with visualizations.

In [ ]:
def label_bedrooms(n):
    n = int(n)
    if n == 0: return 'Studio/Loft'
    if n == 1: return '1 Bedroom'
    return f'{n} Bedrooms'

df['bedroom_label'] = df['bedrooms'].apply(label_bedrooms)

print(df['bedroom_label'].value_counts())

bedroom_label
1 Bedroom      36299
2 Bedrooms      9254
Studio/Loft     4481
3 Bedrooms      1421
4 Bedrooms        92
5 Bedrooms        10
9 Bedrooms        10
8 Bedrooms         2
10 Bedrooms        2
Name: count, dtype: int64


## 6. Final Column Order & Export

### 6.1 Cleaning Summary

Review of all data cleaning and feature engineering steps performed.

| Action | Detail |
|--------|--------|
| Combined | 20 CSV files into one DataFrame |
| Added | `city` and `day_type` columns from filenames |
| Dropped | `Unnamed: 0`, `attr_index`, `rest_index` |
| Renamed | All columns to English snake_case |
| Converted | `person_capacity` to int, booleans to 0/1 |
| Calculated | `price_per_night` (price / 4) |
| Removed | 136 outliers above €500/night |
| Added | `bedroom_label` descriptive column |
| Duplicates | None found |
| Missing values | None found |

### 6.2 Final Column Order

We reorder columns for clarity and analytical workflow:

In [ ]:
column_order = [
    'city', 'day_type', 'room_type', 'is_shared_room', 'is_private_room',
    'is_superhost', 'host_has_multiple_listings', 'host_is_business',
    'person_capacity', 'bedrooms', 'bedroom_label',
    'price', 'price_per_night',
    'cleanliness_rating', 'overall_rating',
    'dist_to_center_km', 'dist_to_metro_km',
    'attraction_index', 'restaurant_index',
    'longitude', 'latitude'
]

df = df[column_order]
print(f'Final shape: {df.shape}')
df.head()

Final shape: (51571, 21)


,city,day_type,room_type,is_shared_room,is_private_room,is_superhost,host_has_multiple_listings,host_is_business,person_capacity,bedrooms,...,price,price_per_night,cleanliness_rating,overall_rating,dist_to_center_km,dist_to_metro_km,attraction_index,restaurant_index,longitude,latitude
0,Amsterdam,Weekend,Private room,0,1,0,0,1,2,1,...,319.640053,79.91,9.0,88.0,4.763360,0.852117,5.871971,11.941560,4.84639,52.34137
1,Amsterdam,Weekend,Private room,0,1,0,0,1,2,1,...,347.995219,87.00,9.0,87.0,5.748310,3.651591,3.985516,8.315410,4.97512,52.36103
2,Amsterdam,Weekend,Private room,0,1,0,0,1,4,2,...,482.975183,120.74,9.0,90.0,0.384872,0.439852,26.116521,76.289005,4.89417,52.37663
3,Amsterdam,Weekend,Private room,0,1,1,0,0,2,1,...,485.552926,121.39,10.0,98.0,0.544723,0.318688,29.270850,71.074937,4.90051,52.37508
5,Amsterdam,Weekend,Entire home/apt,0,0,0,0,0,4,2,...,1001.804420,250.45,9.0,96.0,3.719139,1.196104,5.624209,11.670800,4.86459,52.40175


### 6.3 Export to CSV

Save the cleaned and processed dataset to CSV format for use in Power BI and ML

In [ ]:
output_path = FOLDER + 'airbnb_clean.csv'
df.to_csv(output_path, index=False)
print(f'File saved: airbnb_clean.csv')
print(f'Final dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns')

File saved: airbnb_clean.csv
Final dimensions: 51,571 rows x 21 columns


In [1]:
# Final validation
df_check = pd.read_csv(output_path, nrows=5)
print(f'Columns: {df_check.columns.tolist()}')
print(f'Shape confirmed: {df_check.shape}')

NameError: name 'pd' is not defined